# 04. Evaluation — 모델 평가

## Google Colab 환경 설정

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules or "/kaggle" in sys.executable or os.path.exists("/kaggle/input")

REPO_URL = "https://github.com/Lunarian0928/youth_bio_global_portfolio.git"


def clone_or_pull(repo_dir):
    if os.path.exists(repo_dir):
        get_ipython().system(f"git -C {repo_dir} pull")
    else:
        get_ipython().system(f"git clone {REPO_URL} {repo_dir}")


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    clone_or_pull("youth_bio_global_portfolio")
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations koreanize-matplotlib

elif IN_KAGGLE:
    repo_dir = "/kaggle/working/youth_bio_global_portfolio"
    clone_or_pull(repo_dir)
    os.chdir(f"{repo_dir}/oct_retinal_segmentation")
    sys.path.insert(0, f"{repo_dir}/oct_retinal_segmentation")

    !pip install -q segmentation-models-pytorch albumentations koreanize-matplotlib


## Imports

In [ ]:
import os
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.dataset import OCTRetinalDataset
from src.model import build_model
from src.utils import dice_score, iou_score


## Config

In [ ]:
if IN_COLAB:
    DATA_ROOT = "/content/drive/MyDrive/data/OCT5k"
    MODEL_DIR = "/content/drive/MyDrive/checkpoints"
    SPLITS_DIR = "/content/drive/MyDrive/data/splits"
    RESULTS_DIR = "/content/drive/MyDrive/checkpoints"
elif IN_KAGGLE:
    KAGGLE_BASE = "/kaggle/input/datasets/lunarian0928/oct5k-data/OCT5k"
    DATA_ROOT = f"{KAGGLE_BASE}/data/OCT5k"
    MODEL_DIR = f"{KAGGLE_BASE}/checkpoints"
    SPLITS_DIR = f"{KAGGLE_BASE}/data/splits"
    RESULTS_DIR = "/kaggle/working"
else:
    DATA_ROOT = "../data/OCT5k"
    MODEL_DIR = "../checkpoints"
    SPLITS_DIR = "../data/splits"
    RESULTS_DIR = "../checkpoints"

GRADING = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 평가 대상 모델 목록 (고정 Split + 5-Fold)

In [ ]:
import pandas as pd

MODEL_NAMES = ["fixed", "fold0", "fold1", "fold2", "fold3", "fold4"]
MODEL_PATHS = {name: os.path.join(MODEL_DIR, f"{name}_best.pt") for name in MODEL_NAMES}

for name, path in MODEL_PATHS.items():
    print(name, "->", path, "(존재:", os.path.exists(path), ")")


## 테스트 데이터 로드

In [ ]:
TEST_CSV = os.path.join(SPLITS_DIR, "test.csv")
test_dataset = OCTRetinalDataset.from_csv(TEST_CSV, root_dir=DATA_ROOT)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)
print("테스트 샘플 수:", len(test_dataset))

# 이미지/마스크는 모델과 무관하게 동일하므로 한 번만 모아둔다.
all_images = []
all_masks = []
for images, masks in test_loader:
    all_images.append(images)
    all_masks.append(masks)
all_images = torch.cat(all_images, dim=0)
all_masks = torch.cat(all_masks, dim=0)


## Inference

In [ ]:
all_preds_by_model = {}

for name in MODEL_NAMES:
    model = build_model()
    model.load_state_dict(torch.load(MODEL_PATHS[name], map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    preds_list = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            preds_list.append(preds.cpu())

    all_preds_by_model[name] = torch.cat(preds_list, dim=0)
    print(f"[{name}] 추론 완료")

    del model
    torch.cuda.empty_cache()


## 평가지표 계산 (Dice, IoU, Sensitivity, Specificity)

In [ ]:
NUM_CLASSES = 6
CLASS_NAMES = ["배경", "ILM", "OPL", "IS/OS", "IBRPE", "OBRPE"]


def per_class_metrics(preds, masks):
    rows = []
    for c in range(NUM_CLASSES):
        pred_c = preds == c
        mask_c = masks == c

        tp = (pred_c & mask_c).sum().item()
        fp = (pred_c & ~mask_c).sum().item()
        fn = (~pred_c & mask_c).sum().item()
        tn = (~pred_c & ~mask_c).sum().item()

        dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
        sensitivity = tp / (tp + fn + 1e-8)
        specificity = tn / (tn + fp + 1e-8)

        rows.append(
            {
                "클래스": CLASS_NAMES[c],
                "Dice Score": round(dice, 4),
                "IoU": round(iou, 4),
                "Sensitivity": round(sensitivity, 4),
                "Specificity": round(specificity, 4),
            }
        )
    return rows


# 모델별 클래스 평균 Dice/IoU (고정 vs K-Fold 비교용 스칼라 지표)
model_summary = {}
for name in MODEL_NAMES:
    rows = per_class_metrics(all_preds_by_model[name], all_masks)
    df_rows = pd.DataFrame(rows)
    model_summary[name] = {
        "test_dice": df_rows["Dice Score"].mean(),
        "test_iou": df_rows["IoU"].mean(),
    }
    if name == "fixed":
        df_fixed_detail = df_rows

print("[fixed] 클래스별 상세 결과")
print(df_fixed_detail.to_string(index=False))

fold_names = [n for n in MODEL_NAMES if n != "fixed"]
fold_dices = [model_summary[n]["test_dice"] for n in fold_names]
fold_ious = [model_summary[n]["test_iou"] for n in fold_names]

comparison_test = pd.DataFrame(
    [
        {
            "split": "고정 Split (1회)",
            "test_dice": model_summary["fixed"]["test_dice"],
            "test_dice_std": 0.0,
            "test_iou": model_summary["fixed"]["test_iou"],
            "test_iou_std": 0.0,
        },
        {
            "split": "K-Fold (5-fold 평균)",
            "test_dice": np.mean(fold_dices),
            "test_dice_std": np.std(fold_dices),
            "test_iou": np.mean(fold_ious),
            "test_iou_std": np.std(fold_ious),
        },
    ]
)
print("\n고정 Split vs K-Fold — Test Set 비교")
print(comparison_test.to_string(index=False))

df_fixed_detail.to_csv(os.path.join(RESULTS_DIR, "evaluation_results_fixed.csv"), index=False)
comparison_test.to_csv(os.path.join(RESULTS_DIR, "evaluation_comparison_test.csv"), index=False)
print(f"\n결과 저장 완료: {RESULTS_DIR}")


## 예측 결과 시각화

In [ ]:
VIZ_MODEL = "fixed"  # 시각화 대표 모델
preds_for_viz = all_preds_by_model[VIZ_MODEL]

NUM_SAMPLES = 6
fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(12, NUM_SAMPLES * 4))
col_titles = ["OCT 이미지", "Ground Truth", f"예측 마스크 ({VIZ_MODEL})"]

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=14, fontweight="bold")

for i in range(NUM_SAMPLES):
    image = all_images[i].squeeze().numpy()
    mask = all_masks[i].numpy()
    pred = preds_for_viz[i].numpy()

    axes[i, 0].imshow(image, cmap="gray")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask, cmap="tab10", vmin=0, vmax=5)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred, cmap="tab10", vmin=0, vmax=5)
    axes[i, 2].axis("off")

plt.tight_layout()
VIZ_PATH = os.path.join(RESULTS_DIR, "evaluation_result.png")
plt.savefig(VIZ_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"시각화 저장 완료: {VIZ_PATH}")
